In [ ]:
file_path = "/content/balanced-reviews.txt"

with open(file_path, "r", encoding="utf-16") as f:
    data = f.read()

print(data[:1000])

no	Hotel name	rating	user type	room type	nights	review
2	فندق 72	2	مسافر منفرد	غرفة ديلوكس مزدوجة أو توأم	أقمت ليلة واحدة	“ممتاز”. النظافة والطاقم متعاون. 
3	فندق 72	5	زوج	غرفة ديلوكس مزدوجة أو توأم	أقمت ليلة واحدة	استثنائي. سهولة إنهاء المعاملة في الاستقبال. لاشيئ
16	فندق 72	5	زوج	-	أقمت ليلتين	استثنائي. انصح بأختيار الاسويت و بالاخص غرفه رقم 801. نوعية الارضيه
20	فندق 72	1	زوج	غرفة قياسية مزدوجة	أقمت ليلة واحدة	“استغرب تقييم الفندق كخمس نجوم”. لا شي. يستحق 2 نجمه 
23	فندق 72	4	زوج	غرفة ديلوكس مزدوجة أو توأم	أقمت ليلتين	جيد. المكان جميل وهاديء. كل شي جيد ونظيف بس كان حوض السباحه لايعمل في هذي الفتره حسب كلامهم يقولوا فيه صيانه والله اعلم
24	فندق 72	5	أسرة	غرفة ديلوكس مزدوجة أو توأم	أقمت ليلة واحدة	ممتاز. موقع الفندق ونظافته والاطلاله على البحر وجزيرة  النور التي تحتوي على الفراشات وكذلك قربه من المسجد والممشى  توفر المواقف بجانب الفندق وخدمة صف السيارات. المسبح كان مغلق للصيانه  المواقف تحتاج الى مظلات
25	فندق 72	5	زوج	غرفة ديلوكس مزدوجة أو توأم	أقمت ليلة واحدة	“جيدجداً”. الافطار جيد 

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/balanced-reviews.txt",
    sep="\t",
    encoding="utf-16"
)

df.head()

,no,Hotel name,rating,user type,room type,nights,review
0,2,فندق 72,2,مسافر منفرد,غرفة ديلوكس مزدوجة أو توأم,أقمت ليلة واحدة,“ممتاز”. النظافة والطاقم متعاون.
1,3,فندق 72,5,زوج,غرفة ديلوكس مزدوجة أو توأم,أقمت ليلة واحدة,استثنائي. سهولة إنهاء المعاملة في الاستقبال. ل...
2,16,فندق 72,5,زوج,-,أقمت ليلتين,استثنائي. انصح بأختيار الاسويت و بالاخص غرفه ر...
3,20,فندق 72,1,زوج,غرفة قياسية مزدوجة,أقمت ليلة واحدة,“استغرب تقييم الفندق كخمس نجوم”. لا شي. يستحق ...
4,23,فندق 72,4,زوج,غرفة ديلوكس مزدوجة أو توأم,أقمت ليلتين,جيد. المكان جميل وهاديء. كل شي جيد ونظيف بس كا...


In [ ]:
df = df[df["rating"].isin([1, 2, 4, 5])].copy()

df["label"] = df["rating"].replace({
    1: 0,   # Negative
    2: 0,   # Negative
    4: 1,   # Positive
    5: 1    # Positive
})

print(df["label"].value_counts())

label
0    52849
1    52849
Name: count, dtype: int64


In [ ]:
print("Total reviews:", len(df))

Total reviews: 105698


In [ ]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(f"Training size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")

Training size: 84558
Validation size: 10570
Test size: 10570


In [ ]:
train_df = pd.DataFrame({
    "review": X_train,
    "label": y_train
})

val_df = pd.DataFrame({
    "review": X_val,
    "label": y_val
})

test_df = pd.DataFrame({
    "review": X_test,
    "label": y_test
})

In [ ]:
import pandas as pd

def class_stats(df, name):
    counts = df["label"].value_counts().sort_index()
    perc = df["label"].value_counts(normalize=True).sort_index() * 100

    return pd.DataFrame({
        "Split": name,
        "Class": counts.index,
        "Count": counts.values,
        "Percentage (%)": perc.round(2).values
    })

stats = pd.concat([
    class_stats(train_df, "Train"),
    class_stats(val_df, "Validation"),
    class_stats(test_df, "Test")
], ignore_index=True)

stats

,Split,Class,Count,Percentage (%)
0,Train,0,42279,50.0
1,Train,1,42279,50.0
2,Validation,0,5285,50.0
3,Validation,1,5285,50.0
4,Test,0,5285,50.0
5,Test,1,5285,50.0


In [ ]:
train_df = pd.DataFrame({
    "review": X_train,
    "label": y_train
})

val_df = pd.DataFrame({
    "review": X_val,
    "label": y_val
})

test_df = pd.DataFrame({
    "review": X_test,
    "label": y_test
})

train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

In [ ]:
import os

print(os.listdir())

['.config', 'train.csv', 'validation.csv', 'test.csv', 'balanced-reviews.txt', 'sample_data']


In [ ]:
import pandas as pd

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

def class_stats(df, name):
    counts = df["label"].value_counts().sort_index()
    perc = df["label"].value_counts(normalize=True).sort_index() * 100

    return pd.DataFrame({
        "Split": name,
        "Class": counts.index,
        "Count": counts.values,
        "Percentage (%)": perc.round(2).values
    })

stats = pd.concat([
    class_stats(train_df, "Train"),
    class_stats(val_df, "Validation"),
    class_stats(test_df, "Test")
], ignore_index=True)

stats

,Split,Class,Count,Percentage (%)
0,Train,0,42279,50.0
1,Train,1,42279,50.0
2,Validation,0,5285,50.0
3,Validation,1,5285,50.0
4,Test,0,5285,50.0
5,Test,1,5285,50.0
